# Dimensión de entidades

Construye `data/interim/dimensiones/dim_entidades.parquet` con una fila por entidad financiera y por agrupamiento institucional (códigos `AAxxx`).

Fuentes:
- `Entfin/Nomina.txt`: código y sigla de las entidades activas a la fecha del dump.
- `Entfin/Grupos.txt`: nombres de los agrupamientos institucionales.
- `Info_Hist/Activas/*.txt`: historia de cada entidad activa (fecha de inicio, fusiones absorbidas, cambios de denominación).
- `Info_Hist/Bajas/*.txt`: entidades dadas de baja con sus respectivas leyendas.

Columnas de salida:
- `codigo_entidad`: 5 dígitos para entidades, prefijo `AA` para agrupamientos.
- `nombre`: denominación completa.
- `sigla`: identificador corto (solo para entidades de `Nomina`).
- `estado`: `ACTIVA`, `BAJA` o `AGRUPAMIENTO`.
- `es_agrupamiento`: flag booleano.
- `leyenda_historica`: texto libre con fusiones, cambios de denominación, inicio/baja.

Decisiones metodológicas (ver `docs/notas/metodologia_paneles.md` §3.1, §3.5): se trabaja "bank-as-is" — cada código es una entidad independiente. Los agrupamientos quedan marcados con flag y se tratan como unidades separadas en los paneles de balance.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "utils"))
from paths import RAW, INTERIM, DIMENSIONES, PANELES, REPO

import pandas as pd
import duckdb

IEF_DUMP = RAW / "bcra_ief/202601"
NOMINA = IEF_DUMP / "Entfin/Nomina.txt"
GRUPOS = IEF_DUMP / "Entfin/Grupos.txt"
ACTIVAS_DIR = IEF_DUMP / "Info_Hist/Activas"
BAJAS_DIR = IEF_DUMP / "Info_Hist/Bajas"
OUT = DIMENSIONES / "dim_entidades.parquet"
OUT.parent.mkdir(parents=True, exist_ok=True)

## Nómina y agrupamientos

`Nomina.txt` trae las entidades activas con sigla. `Grupos.txt` trae los agrupamientos.

In [2]:
# Nomina histórica: para que entidades dadas de baja antes del dump corriente conserven su sigla
# (caso típico: HSBC absorbido por Galicia jun-2025; el dump 202601 ya no lo lista). Recorremos
# todos los dumps disponibles y para cada código nos quedamos con la sigla del dump más reciente
# en que apareció.
DUMPS_ROOT = RAW / "bcra_ief"
dumps_disponibles = sorted([d.name for d in DUMPS_ROOT.iterdir()
                            if d.is_dir() and d.name.isdigit() and len(d.name) == 6])
print(f"Recorro {len(dumps_disponibles)} dumps para reconstruir nomina histórica...")

nominas_hist = []
for yyyymm in dumps_disponibles:
    f = DUMPS_ROOT / yyyymm / "Entfin/Nomina.txt"
    if not f.exists():
        continue
    n = pd.read_csv(f, sep="\t", header=None,
                    names=["codigo_entidad", "nombre", "sigla"],
                    encoding="ISO-8859-1", dtype=str)
    n["dump_yyyymm"] = int(yyyymm)
    nominas_hist.append(n)

nomina_full = pd.concat(nominas_hist, ignore_index=True)

# Para cada código, tomamos la fila del dump más reciente en que apareció
nomina = (nomina_full.sort_values("dump_yyyymm", ascending=False)
                     .drop_duplicates(subset="codigo_entidad", keep="first")
                     .drop(columns="dump_yyyymm")
                     .reset_index(drop=True))

# Grupos sigue siendo del dump corriente (no nos importa la historia para el catálogo de agrupamientos)
grupos = pd.read_csv(GRUPOS, sep="\t", header=None,
                     names=["codigo_entidad", "nombre"],
                     encoding="ISO-8859-1", dtype=str)

print(f"Nomina histórica: {len(nomina)} entidades únicas (vs {len(nominas_hist[-1])} en el dump más reciente)  |  Grupos: {len(grupos)} filas")


Recorro 133 dumps para reconstruir nomina histórica...
Nomina histórica: 98 entidades únicas (vs 85 en el dump más reciente)  |  Grupos: 12 filas


## Historia institucional

Cada archivo en `Info_Hist/Activas/` y `Info_Hist/Bajas/` tiene formato:
- Primera línea: cabecera TSV (código, nombre, existe_al_1oct92, período_info, estado).
- Segunda línea en adelante: texto libre con la leyenda histórica (fusiones, cambios de denominación, etc.).

El nombre del archivo incluye el código pegado al nombre; se usa el contenido de la primera línea para evitar depender del parsing del nombre.

In [3]:
def leer_info_hist(directorio):
    registros = []
    for f in sorted(directorio.glob("*.txt")):
        if f.name == "formato.txt":
            continue
        texto = f.read_text(encoding="ISO-8859-1")
        lineas = texto.split("\n", 1)
        cabecera = lineas[0]
        leyenda = lineas[1].strip() if len(lineas) > 1 else ""
        campos = [c.strip('"') for c in cabecera.split("\t")]
        if len(campos) >= 5:
            registros.append({
                "codigo_entidad": campos[0],
                "nombre": campos[1],
                "existe_al_1oct92": campos[2],
                "periodo_info": campos[3],
                "estado": campos[4],
                "leyenda_historica": leyenda,
            })
    return pd.DataFrame(registros)

activas = leer_info_hist(ACTIVAS_DIR)
bajas = leer_info_hist(BAJAS_DIR)
print(f"Activas: {len(activas)}  |  Bajas: {len(bajas)}")
activas.head(3)

Activas: 72  |  Bajas: 225

,codigo_entidad,nombre,existe_al_1oct92,periodo_info,estado,leyenda_historica
0,00007,Banco de Galicia y Buenos Aires S.A.,Existente al 1.10.92,202601,ACTIVA,Al 06/11/1905: Inicio de actividades. Constit...
1,00011,Banco de la Nación Argentina,Existente al 1.10.92,202601,ACTIVA,Al 01/12/1891: Inicio de actividades. Constitu...
2,00014,Banco de la Provincia de Buenos Aires,Existente al 1.10.92,202601,ACTIVA,Al 16/04/2003: Se excluyeron activos y pasivos...


## Ensamblaje

Se une la información en tres bloques (activas, bajas, agrupamientos) y se agrega la sigla cuando existe.

**Observación sobre códigos reutilizados**: el BCRA reasigna códigos de entidad a lo largo del tiempo. Hay casos donde el mismo código identifica a una entidad dada de baja y, más tarde, a otra entidad distinta. Por ejemplo, el código `00001` fue Banco 1784 y posteriormente Deutsche Bank. Por eso la clave primaria de esta dimensión es compuesta: `(codigo_entidad, nombre)`. Para joins con el panel de balance (donde `codigo_entidad` es la única columna disponible), la entidad que corresponde es la que estaba activa en ese `yyyymm`. La columna `es_vigente` marca la entidad actualmente activa por código, para simplificar el join cuando se trabaja con datos recientes.

In [4]:
activas["es_agrupamiento"] = False
bajas["es_agrupamiento"] = False

agrup = grupos.copy()
agrup["estado"] = "AGRUPAMIENTO"
agrup["existe_al_1oct92"] = pd.NA
agrup["periodo_info"] = pd.NA
agrup["leyenda_historica"] = pd.NA
agrup["es_agrupamiento"] = True

# Entidades en Nomina que no están en Info_Hist (típicamente entidades nuevas o sin
# leyenda histórica registrada). Se agregan como ACTIVA sin leyenda.
en_info_hist = set(activas.codigo_entidad) | set(bajas.codigo_entidad)
solo_nomina = nomina[~nomina.codigo_entidad.isin(en_info_hist) & ~nomina.codigo_entidad.str.startswith("AA")].copy()
solo_nomina["estado"] = "ACTIVA"
solo_nomina["existe_al_1oct92"] = pd.NA
solo_nomina["periodo_info"] = pd.NA
solo_nomina["leyenda_historica"] = pd.NA
solo_nomina["es_agrupamiento"] = False

dim = pd.concat([activas, bajas, agrup, solo_nomina[activas.columns]], ignore_index=True)
dim = dim.merge(nomina[["codigo_entidad", "sigla"]], on="codigo_entidad", how="left")

# La entidad vigente por código es la ACTIVA o la AGRUPAMIENTO; si hay múltiples bajas con el
# mismo código, ninguna queda marcada como vigente (ya no existe esa entidad con ese código).
dim["es_vigente"] = dim["estado"].isin(["ACTIVA", "AGRUPAMIENTO"])

columnas = ["codigo_entidad", "nombre", "sigla", "estado", "es_vigente",
            "es_agrupamiento", "existe_al_1oct92", "periodo_info", "leyenda_historica"]
dim = dim[columnas]
dim.head()

,codigo_entidad,nombre,sigla,estado,es_vigente,es_agrupamiento,existe_al_1oct92,periodo_info,leyenda_historica
0,00007,Banco de Galicia y Buenos Aires S.A.,GALICIA,ACTIVA,True,False,Existente al 1.10.92,202601,Al 06/11/1905: Inicio de actividades. Constit...
1,00011,Banco de la Nación Argentina,NACION,ACTIVA,True,False,Existente al 1.10.92,202601,Al 01/12/1891: Inicio de actividades. Constitu...
2,00014,Banco de la Provincia de Buenos Aires,PRBSAS,ACTIVA,True,False,Existente al 1.10.92,202601,Al 16/04/2003: Se excluyeron activos y pasivos...
3,00015,Industrial and Commercial Bank of China (Argen...,INDUSTR,ACTIVA,True,False,Alta posterior al 1.10.92,202601,Al 02/05/2006: Inicio de actividades sobre la ...
4,00016,Citibank N.A.,CITIBAN,ACTIVA,True,False,Existente al 1.10.92,202601,Al 10/11/1914: Inicio de actividades. Constitu...


## Persistencia

In [5]:
dim.to_parquet(OUT, index=False)
print(f"Escrito: {OUT.relative_to(REPO)}")
print(f"Filas: {len(dim):,}")

Escrito: data/interim/dimensiones/dim_entidades.parquet
Filas: 312


## Validaciones

In [6]:
# No hay clave simple única: el BCRA puede reasignar códigos a nuevas entidades (el caso más extremo
# es Banco Macro, donde código y nombre coinciden entre una entidad dada de baja en 2003 y la actual).
# La unicidad de la fila se garantiza con un índice sintético.
dim = dim.reset_index(drop=True)
dim["idx"] = dim.index

assert dim["estado"].isin(["ACTIVA", "BAJA", "AGRUPAMIENTO"]).all(), "Hay estados inesperados"
assert dim["es_agrupamiento"].sum() == len(grupos), "El flag de agrupamiento no coincide con Grupos.txt"

# Para que el panel de balance se pueda joinear por código, tiene que haber UNA entidad vigente por código.
vigentes_por_codigo = dim[dim["es_vigente"]].groupby("codigo_entidad").size()
assert (vigentes_por_codigo <= 1).all(), "Hay códigos con más de una entidad marcada como vigente"

# Re-guardo con la columna idx incluida
dim.to_parquet(OUT, index=False)

resumen = dim["estado"].value_counts().to_dict()
n_codigos_reutilizados = (dim.groupby("codigo_entidad").size() > 1).sum()
print("Validaciones OK")
for k, v in resumen.items():
    print(f"  {k}: {v}")
print(f"  Códigos reutilizados (>1 entidad histórica): {n_codigos_reutilizados}")

Validaciones OK
  BAJA: 225
  ACTIVA: 75
  AGRUPAMIENTO: 12
  Códigos reutilizados (>1 entidad histórica): 22


## Muestra con DuckDB

In [7]:
duckdb.sql(f"""
    select codigo_entidad, nombre, sigla, estado
    from '{OUT}'
    where estado = 'ACTIVA'
    order by codigo_entidad
    limit 15
""").df()

,codigo_entidad,nombre,sigla,estado
0,00007,Banco de Galicia y Buenos Aires S.A.,GALICIA,ACTIVA
1,00011,Banco de la Nación Argentina,NACION,ACTIVA
2,00014,Banco de la Provincia de Buenos Aires,PRBSAS,ACTIVA
3,00015,Industrial and Commercial Bank of China (Argen...,INDUSTR,ACTIVA
4,00016,Citibank N.A.,CITIBAN,ACTIVA
5,00017,Banco BBVA Argentina S.A.,BBVA.AR,ACTIVA
6,00020,Banco de la Provincia de Córdoba S.A.,CORDOBA,ACTIVA
7,00027,Banco Supervielle S.A.,SUPERVI,ACTIVA
8,00029,Banco de la Ciudad de Buenos Aires,CIUDAD,ACTIVA
9,00034,Banco Patagonia S.A.,PATAGON,ACTIVA
